# 09 – Alert Generation

In this notebook I combine two things:

1. **Lake risk** from my final model (which lakes are more susceptible)
2. **Weather trigger** from a free live forecast (Open-Meteo): rainfall and temperature for the next 3 days

Together they give an alert level: **Normal / Watch / Warning / Danger**.

> The weather thresholds below are **demo values** chosen for this prototype, not official PMD/NDMA thresholds. This is a student prototype, not an official warning system.

### 1. Settings (demo thresholds)

In [4]:
RISK_HIGH_PCT = 0.95     # top 5% of lakes by model score = High risk
RISK_MEDIUM_PCT = 0.80   # next 15% = Medium risk

# DEMO weather thresholds (not official)
RAIN_ELEVATED_MM = 20    # rain in next 3 days
RAIN_EXTREME_MM = 50
TEMP_ELEVATED_C = 15     # max temperature in next 3 days (melt)
TEMP_EXTREME_C = 25

### 2. Lake risk from my final model

In [5]:
import pandas as pd
import numpy as np
import joblib, time, requests
from pathlib import Path

PROJECT_ROOT = Path("..")
saved = joblib.load(PROJECT_ROOT / "artifacts" / "final_model.joblib")
model, features = saved["pipeline"], saved["features"]
lakes = pd.read_csv(PROJECT_ROOT / "data" / "final" / "glofguard_training_data.csv")

lakes["risk_score"] = model.predict_proba(lakes[features])[:, 1]
pct = lakes["risk_score"].rank(pct=True)
lakes["risk_tier"] = np.select([pct > RISK_HIGH_PCT, pct > RISK_MEDIUM_PCT], ["High", "Medium"], "Low")
lakes["risk_tier"].value_counts()

risk_tier
Low       7044
Medium    1321
High       441
Name: count, dtype: int64

### 3. Live weather forecast (Open-Meteo, free, no key)

I only fetch weather for High and Medium risk lakes, because Low risk lakes stay Normal in my alert rules.

In [6]:
def fetch_weather(df, batch=50, tries=4):
    rows = []
    for start in range(0, len(df), batch):
        part = df.iloc[start:start + batch]
        params = {
            "latitude": ",".join(part["latitude"].round(4).astype(str)),
            "longitude": ",".join(part["longitude"].round(4).astype(str)),
            "daily": "precipitation_sum,temperature_2m_max",
            "forecast_days": 3,
            "timezone": "Asia/Karachi",
        }
        data = None
        for attempt in range(tries):
            r = requests.get("https://api.open-meteo.com/v1/forecast", params=params, timeout=60)
            if r.ok:
                data = r.json()
                break
            print(f"Batch {start}: error {r.status_code}, retrying...")
            time.sleep(5 * (attempt + 1))
        if data is None:
            print(f"Batch {start}: skipped")
            continue
        data = data if isinstance(data, list) else [data]
        for sid, d in zip(part["sample_id"], data):
            rows.append({
                "sample_id": sid,
                "rain_next3_mm": float(np.nansum(np.array(d["daily"]["precipitation_sum"], dtype=float))),
                "tmax_next3_c": float(np.nanmax(np.array(d["daily"]["temperature_2m_max"], dtype=float))),
                "forecast_start": d["daily"]["time"][0],
            })
        time.sleep(1)
    return pd.DataFrame(rows)

watch = lakes[lakes["risk_tier"] != "Low"]
weather = fetch_weather(watch)
print("Weather fetched for", len(weather), "of", len(watch), "lakes")
weather.describe()

Batch 600: error 429, retrying...
Batch 600: error 429, retrying...
Batch 600: error 429, retrying...
Batch 1200: error 429, retrying...
Batch 1200: error 429, retrying...
Batch 1200: error 429, retrying...
Weather fetched for 1762 of 1762 lakes


,sample_id,rain_next3_mm,tmax_next3_c
count,1762.000000,1762.000000,1762.000000
mean,4925.208286,4.466856,-0.739671
std,2067.303972,4.879075,3.092250
min,141.000000,0.000000,-9.000000
25%,4037.250000,2.000000,-2.900000
50%,5813.500000,3.700000,-1.000000
75%,6650.750000,4.600000,1.200000
max,8806.000000,58.000000,15.800000


### 4. Alert rules

| Lake risk \\ Weather | Normal | Elevated | Extreme |
|---|---|---|---|
| High | Watch | Warning | Danger |
| Medium | Normal | Watch | Warning |
| Low | Normal | Normal | Normal |

In [7]:
df = lakes.merge(weather, on="sample_id", how="left")

rain, temp = df["rain_next3_mm"].fillna(0), df["tmax_next3_c"].fillna(-99)
df["weather_level"] = np.select(
    [(rain >= RAIN_EXTREME_MM) | (temp >= TEMP_EXTREME_C),
     (rain >= RAIN_ELEVATED_MM) | (temp >= TEMP_ELEVATED_C)],
    ["Extreme", "Elevated"], "Normal")

ALERT = {
    ("High", "Normal"): "Watch",   ("High", "Elevated"): "Warning",  ("High", "Extreme"): "Danger",
    ("Medium", "Normal"): "Normal", ("Medium", "Elevated"): "Watch", ("Medium", "Extreme"): "Warning",
}
df["alert"] = [ALERT.get((r, w), "Normal") for r, w in zip(df["risk_tier"], df["weather_level"])]

order = ["Danger", "Warning", "Watch", "Normal"]
print(df["alert"].value_counts().reindex(order, fill_value=0))

alert
Danger        0
Warning      12
Watch       491
Normal     8303
Name: count, dtype: int64


### 5. Lakes that need attention

In [8]:
cols = ["sample_id", "latitude", "longitude", "risk_score", "risk_tier",
        "rain_next3_mm", "tmax_next3_c", "weather_level", "alert"]
attention = (df[df["alert"] != "Normal"][cols]
             .assign(rank=lambda d: d["alert"].map({a: i for i, a in enumerate(order)}))
             .sort_values(["rank", "risk_score"], ascending=[True, False])
             .drop(columns="rank"))
attention.head(20)

,sample_id,latitude,longitude,risk_score,risk_tier,rain_next3_mm,tmax_next3_c,weather_level,alert
6983,6984,36.109904,74.247869,0.995740,High,0.8,15.8,Elevated,Warning
7029,7030,36.532562,73.152976,0.936592,High,39.2,1.8,Elevated,Warning
6042,6043,36.644067,73.646150,0.934897,High,20.5,-2.8,Elevated,Warning
6044,6045,36.561408,73.595354,0.897586,High,32.3,1.2,Elevated,Warning
7040,7041,36.564265,73.139619,0.895996,High,39.2,1.0,Elevated,Warning
6065,6066,36.622973,73.706453,0.891428,High,20.5,-3.4,Elevated,Warning
6066,6067,36.622706,73.706366,0.891305,High,20.5,-3.4,Elevated,Warning
6075,6076,36.632493,73.796539,0.877647,High,20.5,-3.9,Elevated,Warning
6074,6075,36.629681,73.750280,0.872234,High,20.5,-4.3,Elevated,Warning
330,331,36.308984,71.980904,0.633335,Medium,58.0,1.7,Extreme,Warning


### 6. Save today's alerts (for the UI)

In [9]:
out = df[cols + ["forecast_start"]].copy()
out["risk_score"] = (out["risk_score"] * 100).round(1)
out.to_csv(PROJECT_ROOT / "artifacts" / "alerts_latest.csv", index=False)
print("Saved: artifacts/alerts_latest.csv")

Saved: artifacts/alerts_latest.csv


### Conclusion

I combined my model's lake risk with a live 3-day weather forecast to create alerts. On the run of 25 October 2026, I got **0 Danger, 12 Warning, 491 Watch and 8,303 Normal** lakes.

- Most alerts are **Watch**, because every High risk lake gets at least Watch, even in normal weather.
- The 12 **Warning** lakes were High risk lakes with heavy forecast rain (20–39 mm) or warm temperatures (15.8°C), and three Medium risk lakes near 72°E with extreme rain (58 mm).
- No lake reached **Danger**, which makes sense for cold, mostly dry late-October weather.

This shows the alert system reacts to both lake risk and weather, and the results change every day with the forecast.

**Note:** the weather thresholds are demo values, not official PMD/NDMA thresholds. The forecast is for the grid point near each lake. The risk score comes from a model trained on proxy labels (see notebooks 06–08), so an alert means "check this lake", not "this lake will burst".